# Instalando as Bibliotecas Necessárias



In [ ]:
# Installing the numpy, netcdf4, boto3 and gdal libraries
!pip install -q cartopy boto3 gdal salem rasterio pyproj geopandas descartes ultraplot geobr

# download do arquivo "utilities_goes16e19.py"
!wget -c https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/utilities_goes16e19.py

# download da paleta de cores para o canal do infravermelho
!wget -c https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/ir.cpt

# Monta drive
from google.colab import drive
drive.mount('/content/drive')

# Caminho do diretório
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/4_SATELITE/codigos_imagem_satelite_REFERENCIA'

# Diretório de Saída
import os
dir_output = f'{dir}/output/SAT_07'
os.makedirs(dir_output, exist_ok=True)

# Várias Imagens em Projeção Satélite T-Realçada + Flashes do GLM

In [ ]:
#========================================================================================================================#
#                                          IMPORTAÇÃO DAS BIBLIOTECAS
#========================================================================================================================#
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
import cartopy, cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from datetime import timedelta, datetime
from utilities_goes16e19 import download_CMI, download_GLM, remap, loadCPT
import numpy as np
import os
import pandas as pd
import geopandas as gpd
import salem
import geobr

#========================================================================================================================#
#                                        CRIA DIRETÓRIO DE ENTRADA E SAÍDA
#========================================================================================================================#
input = "/content/input"; os.makedirs(input, exist_ok=True)
output = dir_output

#========================================================================================================================#
#                                        DEFINE OS LIMITES DA IMAGEM
#========================================================================================================================#
# canal
band = '13'

# área desejada da imagem
lonmin, lonmax, latmin, latmax = -48, -43, -23.5, -20

# # coloca os limites da área numa lista
extent = [lonmin, latmin, lonmax, latmax]

#========================================================================================================================#
#                                              CARREGA SHAPEFILES
#========================================================================================================================#
# carrega o shapefiles de todos municípios do Brasil
shapefile_municipios = geobr.read_municipality(year=2020)

# filtra apenas o município de Campina Grande
shapefile_municipio = shapefile_municipios[shapefile_municipios['name_muni'] == 'Itajubá']

# carrega o shapefile dos estados brasileiros. Usa a função read_state do geobr para obter os polígonos dos estados
shapefile_estados = geobr.read_state(year=2020)

# filtra os estados dentro da extensão do mapa. Cria uma máscara para selecionar apenas os estados que intersectam com a área do mapa
shapefile_estados_filtrados = shapefile_estados.cx[extent[0]:extent[2], extent[1]:extent[3]]

#========================================================================================================================#
#                                              LOOP DAS IMAGENS
#========================================================================================================================#
# Loop das imagens
for date_image in pd.date_range('202601242200', '202601250030', freq='10min'):

    #--------------------------------------------------------------------------#
    #                          DATA E HORÁRIO
    #--------------------------------------------------------------------------#
    # data
    yyyymmddhhmn = date_image.strftime('%Y%m%d%H%M') # '202404301300'

    # define o satélite: GOES-16 ou GOES-19
    start_g19 = datetime(2025,4,7,0,0)
    imagem_atual = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M')
    goes_number = '16' if imagem_atual < start_g19 else '19'

    # extrai o ano, mês, dia, hor e min
    yyyy = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%Y')
    mm = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%m')
    dd = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%d')
    hh = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%H')
    mn = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%M')

    print('#=====================================================================================================#')
    print(f'                          PROCESSANDO A IMAGEM = {yyyy}-{mm}-{dd} {hh}{mn} UTC'                       )
    print('#=====================================================================================================#')

    # download do arquivo
    file_name = download_CMI(yyyymmddhhmn, band, goes_number, input)

    # caminho do arquivo que foi baixado
    path = f'{input}/{file_name}.nc'

    #--------------------------------------------------------------------------#
    #                    REPROJETA OS DADOS DO ABI
    #--------------------------------------------------------------------------#
    # chama a função que faz a reprojeção (file, variable, extent, resolution)
    grid = remap(path, 'CMI', extent, 2)

    # leitura do dado e transforma para °C
    data = grid.ReadAsArray() - 273.15

    #--------------------------------------------------------------------------#
    #                       BAIXA OS DADOS DO GLM
    #--------------------------------------------------------------------------#
    # Data da imagem atual e da próxima
    date_ini = str(datetime(int(yyyy),int(mm),int(dd),int(hh),int(mn)))
    date_end = str(datetime(int(yyyy),int(mm),int(dd),int(hh),int(mn)) + timedelta(minutes=10))
    date_loop = date_ini

    # loop de cumulação do GLM
    lats_flash, lons_flash = np.array([]), np.array([])
    while (date_loop <= date_end):

        # data
        yyyymmddhhmnss = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y%m%d%H%M%S')

        # Download o arquivo
        file_glm = download_GLM(yyyymmddhhmnss, goes_number, input)

        # Verifica se o arquivo existe antes de processar
        file_path = f'{input}/{file_glm}.nc'
        if os.path.exists(file_path):

            # leitura do arquivo
            glm_20s = xr.open_dataset(file_path)

            # appenda as lats / longs
            lats_flash = np.append(lats_flash, glm_20s['flash_lat'][:])
            lons_flash = np.append(lons_flash, glm_20s['flash_lon'][:])

            # fecha o arquivo
            glm_20s.close()
        else:
            print(f"Arquivo não encontrado: {file_path}")

        # incrementa a variável the date_loop
        date_loop = str(datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S') + timedelta(seconds=20))
    #--------------------------------------------------------------------------#

    # coloca os flashes num dataframe
    data_flash = {'lat': lats_flash, 'lon': lons_flash}
    df = pd.DataFrame(data_flash)

    # seleciona os flahes da região de interesee
    df_flash_filtered = df[ (df['lon'] > lonmin) & (df['lon'] < lonmax) & (df['lat'] > latmin) & (df['lat'] < latmax)]

    # transforma o dataframe para array
    lons_flash_filtered, lats_flash_filtered = df_flash_filtered['lon'].values, df_flash_filtered['lat'].values

    #--------------------------------------------------------------------------#
    #                           PLOTA A IMAGEM
    #--------------------------------------------------------------------------#
    # tamanho da figura (largura x altura em polegadas)
    plt.figure(figsize=(14,11))

    # projeção geoestacionária do cartopy
    ax = plt.axes(projection=ccrs.PlateCarree())

    # converte o arquivo CPT para ser usado em Python
    cpt = loadCPT('ir.cpt')
    cmap = cm.colors.LinearSegmentedColormap('cpt', cpt)

    # plota imagem
    img = ax.imshow(data, origin='upper', vmin=-103.0, vmax=84, extent=[lonmin, lonmax, latmin, latmax], cmap=cmap, alpha=1.0)

    # legenda
    ax.legend(loc='lower right', ncols=1, facecolor='white', frameon=True)

    # linhas costeiras, bordas e linhas de grade do mapa
    gl = ax.gridlines(crs=ccrs.PlateCarree(), color='gray', alpha=1.0, linestyle='--', linewidth=0.25, xlocs=np.arange(-180, 180, 2), ylocs=np.arange(-90, 90, 1), draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False

    # plota o município
    shapefile_municipio.plot(ax=ax, edgecolor='cyan', facecolor='none', linewidth=2.0, label='Santa Maria', zorder=3)

    # plota estados
    shapefile_estados_filtrados.plot(ax=ax, edgecolor='white', facecolor='none', linewidth=1.0, zorder=3)

    # define os limites do eixo para a extensão da imagem
    ax.set_xlim(extent[0], extent[2])  # lonmin, lonmax
    ax.set_ylim(extent[1], extent[3])  # latmin, latmax

    # plota glm
    glm = plt.scatter(lons_flash_filtered, lats_flash_filtered, transform=ccrs.PlateCarree(), marker='o', s=20, facecolor='white', edgecolor='black', linewidth=1, alpha=0.8, zorder=3, label=f'Flashes={str(len(lats_flash_filtered)).zfill(4)}')

    # barra de cores
    plt.colorbar(img, label='Temperatura de Brilho (°C)', extend='both', orientation='horizontal', pad=0.05, fraction=0.060)

    # leitura da data/horário do arquivo NetCDF como uma string
    date = (datetime.strptime(xr.open_dataset(path).time_coverage_start, '%Y-%m-%dT%H:%M:%S.%fZ')).strftime('%Y-%m-%d %H:%M UTC')

    # título da figura
    plt.title(f'GOES-{goes_number} Banda 13 (10.3 µm) + GLM Flashes\nABI: {date}', fontweight='bold', fontsize=10, loc='left')
    plt.title(f'GLM: {str(date_ini)} - {str(date_end)}', fontsize="10", loc="right")

    # salva figura
    plt.savefig(f'{output}/SAT_07_G{goes_number}_ch{band}_retangular_trealcada_flashes_{yyyy}-{mm}-{dd}_{hh}:{mn}_UTC.jpg', bbox_inches='tight', dpi=300)

    # mostra figura na tela (descomente aqui para mostrar cada imagem)
    #plt.show()
    print('\n')